In [5]:
from pathlib import Path
import pandas as pd

recordings_dir = Path.cwd()  # eval.ipynb is in data/recordings
csv_files = sorted(recordings_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {recordings_dir}")

all_frames = []
for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    df["source_file"] = csv_path.name

    # Parse ik_counts strings like "8;8" into numeric totals for statistics.
    def parse_ik_total(value):
        try:
            parts = str(value).split(";")
            nums = [int(p.strip()) for p in parts if p.strip() != ""]
            return sum(nums) if nums else float("nan")
        except Exception:
            return float("nan")

    df["ik_total"] = df["ik_counts"].apply(parse_ik_total)
    all_frames.append(df)

all_data = pd.concat(all_frames, ignore_index=True)

per_file_summary = (
    all_data.groupby("source_file", as_index=False)
    .agg(
        latency_avg_ms=("latency_ms", "mean"),
        latency_median_ms=("latency_ms", "median"),
        latency_min_ms=("latency_ms", "min"),
        latency_max_ms=("latency_ms", "max"),
        compute_avg_ms=("compute_ms", "mean"),
        compute_median_ms=("compute_ms", "median"),
        compute_min_ms=("compute_ms", "min"),
        compute_max_ms=("compute_ms", "max"),
        ik_total_avg=("ik_total", "mean"),
        ik_total_median=("ik_total", "median"),
        ik_total_min=("ik_total", "min"),
        ik_total_max=("ik_total", "max"),
    )
    .sort_values("source_file")
)

summary_text = per_file_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}")

print("Per-file statistics (avg/median/min/max):")
print(summary_text)

output_path = recordings_dir / "eval.txt"
with output_path.open("w", encoding="utf-8") as f:
    f.write("Per-file statistics (avg/median/min/max):\n")
    f.write(summary_text)
    f.write("\n")

print(f"\nSaved summary to: {output_path}")

Per-file statistics (avg/median/min/max):
                                        source_file  latency_avg_ms  latency_median_ms  latency_min_ms  latency_max_ms  compute_avg_ms  compute_median_ms  compute_min_ms  compute_max_ms  ik_total_avg  ik_total_median  ik_total_min  ik_total_max
10_deg_either_side_no_collision_20260313_160446.csv         28.5038            28.1770         24.5330         48.7490         28.3845            28.0690         24.4260         48.6320       80.0000          80.0000            80            80
 2_deg_either_side_no_collision_20260313_155139.csv          2.9930             2.9130          1.0960          5.0430          2.8727             2.7970          1.0910          4.7820       15.9958          16.0000             8            16
       no_rotation_no_collision_20260313_141504.csv          2.7301             2.6520          1.0680          4.9760          2.6084             2.5300          1.0630          4.8510       15.9958          16.0000       